# Homology Basis Loop Comparison

Purpose: We look at the error and runtime for Myles meshes with both minimal and maximal length homology basis loops to determine whether we get substantial improvement with shorter loops.

In [ ]:
import os, sys
import numpy as np
import igl

In [ ]:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
file_dir = './'
moudle_dir = os.path.join(file_dir)
sys.path.append(file_dir)
import penner

First, we need a mesh to process. We assume an input mesh given as a collection of two matrices:
- `V`: a $\#V \times 3$ matrix determining geometry, where each row stores the position in space (as x, y, z coordinates) of a mesh vertex
- `F`: a $\#F \times 3$ matrix determining connectivity, where each row stores the vertex indices of a mesh face

Meshes of this form can be easily stored and loaded with the `obj` file format

In [13]:
data_dir = os.path.join('..', '..', 'data', 'closed-Myles')
V, F = igl.read_triangle_mesh(os.path.join(data_dir, 'fertility_tri.obj'))

## Seamless Parametrization

Our method can reliably produce a seamless parametrization for closed manifold meshes. In order to achieve robust seamless parametrization, it is necessary to change the mesh connectivity. Our method produces a refinement `V_r`, `F_r` of the original mesh, along with a cut to disk connectivity `FT_r` and uv coordinates `uv_r` for this cut mesh.

We also output target directions `Du`, `Dv` for the faces of the refined mesh. These are necessary for further optimization of the parametrization (see below).

In [14]:
field_params = penner.FieldParameters()
alg_params = penner.NewtonParameters()
V_r, F_r, uv_r, FT_r, Du, Dv = penner.parametrize_seamless(V, F, field_params, alg_params)

[2026-07-15 23:39:42.812] [info] 1/27954 faces fixed
[2026-07-15 23:39:43.421] [info] 461/27954 faces fixed


solve_cone_rounding *** Statistics of MiSo Solver ***
		 Number of CG    iterations  = 0
		 Number of LOCAL iterations  = 1707
		 Number of FULL  iterations  = 2
		 Number of ROUNDING          = 14438
		 time searching next integer = 0.363846s



[2026-07-15 23:39:44.445] [info] 1/27954 faces fixed
[2026-07-15 23:39:44.477] [info] Gauss-Bonnet error before cone removal: 2.2846506908535957e-09
[2026-07-15 23:39:44.477] [info] Gauss-Bonnet error after cone removal: 2.2846506908535957e-09
[2026-07-15 23:39:44.605] [info] Writing energy data to iteration_energy_log.csv
[2026-07-15 23:39:44.606] [info] Writing stability data to iteration_stability_log.csv
[2026-07-15 23:39:44.687] [info] itr(0) lm(1) max_error(3.069734796627816))
[2026-07-15 23:39:44.934] [info] itr(1) lm(0.5) max_error(1.618755830960783) rmsre (0.06580136989604664)
[2026-07-15 23:39:45.181] [info] itr(2) lm(0.5) max_error(0.8444606579320171) rmsre (0.09868487255928819)
[2026-07-15 23:39:45.466] [info] itr(3) lm(0.5) max_error(0.42597833272700747) rmsre (0.11512348176811338)
[2026-07-15 23:39:45.719] [info] itr(4) lm(0.5) max_error(0.20405692373548234) rmsre (0.12355255501195478)
[2026-07-15 23:39:45.941] [info] itr(5) lm(1) max_error(0.046115636845737384) rmsre (0.

The Penner coordinate methods provide a numerically seamless parametrization satisfying seamless and feature alignment constraints. Generally, these parametrizations are already approximately isometric and high quality. However, they can be further optimized and also made to align with the guiding field directions.

We provide methods to perform this optimization directly in terms of the uv coordinates. We generally use our Penner coordinate parametrizations as input, but any seamless parametrization could be used here.

In [15]:
uv_param = penner.load_parameters("../app/symdir.json")
uv_r = penner.optimize_seamless_parameterization(V_r, F_r, uv_r, FT_r, Du, Dv, uv_param)

Number of threads: 12
[2026-07-15 23:39:53.754] [info] 2402 boundary faces
[2026-07-15 23:39:53.898] [info] built map of 15179 vertices
[2026-07-15 23:39:53.904] [info] Building constraints
[2026-07-15 23:39:54.038] [info] Adding 0 misalignment constraints
[2026-07-15 23:39:54.039] [info] 2402 seam vertices
[2026-07-15 23:39:54.039] [info] computing degree for mesh with 13971 vertices
[2026-07-15 23:39:54.044] [info] 1305 independent vertices
[2026-07-15 23:39:54.044] [info] 1097 dependent vertices
[2026-07-15 23:39:54.045] [info] partially reduced system matrix has 34814 nonzeros
[2026-07-15 23:39:54.047] [info] Eliminating constraints
[2026-07-15 23:39:54.047] [info] Factorizing 2404x28164 matrix
[2026-07-15 23:39:54.047] [info] 15610 nonzeros in A
[2026-07-15 23:39:54.056] [info] Rank is 210
[2026-07-15 23:39:54.057] [info] 1552 nonzeros in R
[2026-07-15 23:39:54.057] [info] Factorizaiton complete
[2026-07-15 23:39:54.057] [info] Matrix rank: 210
[2026-07-15 23:39:54.057] [info] Rem

This seamless parametrization can be visualized with the following viewer

In [ ]:
penner.view_seamless_parameterization(V_r, F_r, uv_r, FT_r, "seamless", True)

We infer the seamless constraints for our parametrization using a guiding cross field. We use a facet-based cross field, with four rotationally symmetric vectors in each face.

The cross field is represented by a principal direction $d_0$ per face, where the remaining directions $d_1, d_2, d_3$ determined by symmetry. We also explicitly encode the period jumps across edges. A period jump of $i$ across an edge implies that $d_k$ in one face corresponds to the direction $d_{(k + i) \mod 4}$ in the adjacent face. These period jumps determine the singularities and general topology of the cross field.

We provide an implementation to optimize a smooth cross field on the surface using the mixed integer formulation of Bommes et al. 2009.

The field is represented with the following matrices:
- `reference_field`: a $\#F \times 3$ matrix encoding reference directions $d_r$ on the face, where each row stores the coordinates of a direction on a face
- `theta`: a length $\#F$ vector of angles, where each angle corresponds to a counterclockwise rotation in the plane of the face. Rotating the reference direction $d_r$ in the face by this angle gives the representative direction $d_0$ of the smooth cross field. The other three directions are obtained by additional rotations of $\pi/2$
- `kappa`: a $\#F \times 3$ matrix storing the intrinsic rotation of the reference field directions across the edge opposite the triangle corner. While this angle can be inferred directly from the reference field, there is ambiguity due to periodicity. Rather than using a fragile convention, we store the angles explicitly.
- `period_jump`: a $\#F \times 3$ matrix storing period jumps across the edge opposite the triangle corner.

We also provide custom functions to read and write these matrices to file

In [ ]:
field_params = penner.FieldParameters()
reference_field, theta, kappa, period_jump = penner.generate_frame_field(V, F, field_params)

The following function can be used to view the guiding cross field

In [ ]:
penner.view_cross_field(V, F, reference_field, theta, kappa, period_jump, "field")

If a target field is provided, it can be parametrized with the following method.

Since the parametrization may refine the original mesh, this method also refines the input field to produce an equivalent field on the refined mesh.

In [ ]:
alg_params = penner.NewtonParameters()
param_data = penner.generate_seamless_parametrization(V, F, reference_field, theta, kappa, period_jump, alg_params)
V_r, F_r, uv_r, FT_r, reference_field_r, theta_r, kappa_r, period_jump_r = param_data

The following methods explore the inner workings of our approach.

The core data structure of this library is a halfedge mesh representation that stores differentiable intrinsic metric coordinates (in particular, Penner-coordinates).

These meshes also explicitly store the desired constraints. The following function constructs a differentiable metric to satisfy seamless constraints inferred from the cross field.

In [ ]:
marked_metric_params = penner.MarkedMetricParameters()
marked_metric, vtx_reindex, rotation_form, Th_hat = penner.generate_metric_from_field(
  V,
  F,
  theta,
  kappa,
  period_jump,
  marked_metric_params)

The core method of this library is a modified Newton's method that seeks a root of the nonlinear constraints, e.g., seamless constraints. The following method runs this optimization and constructs a refinement of the original mesh that satisfies these constraints.

The following data describes the relationship of the refined mesh to the original mesh:
- `fn_to_f`: a map from the faces of the refined mesh to the parent face of the original mesh. Note that each refined mesh face has a single, well-defined parent.
- `endpoints`: a map from vertices of the refined mesh to pairs of vertices of the original mesh. For vertices in a refined edge, these are the endpoints of the edge containing the vertex. For vertices already present in the original mesh, this is a dummy value. We do not introduce any vertices in the interiors of faces.

These maps are sufficient to reconstruct essentially all element correspondences of the refined mesh. For instance, these maps can be used to refine the input cross field.

In [ ]:
alg_params = penner.NewtonParameters()
V_r, F_r, uv_r, FT_r, fn_to_f, endpoints = penner.parametrize_seamless_metric(V, F, marked_metric, alg_params)

## Feature Aligned Parametrization

For surfaces with prominent feature edges, such as crease edges in mechanical models, it is essential to produce parametrizations that are aligned to these features. This enables quad meshes and spline surfaces constructed from these parametrizations to preserve these feature edges.

We open a mesh with prominent feature edges.

In [ ]:
data_dir = os.path.join('..', '..', 'data', 'closed-Myles')
V, F = igl.read_triangle_mesh(os.path.join(data_dir, 'fandisk.obj'))

The following method produces a feature aligned seamless parametrization for closed manifold meshes. The feature edges are determined automatically using a simple dihedral angle test, with some pruning to remove spurious isolated feature edges. 

As with basic seamless parametrization, it is necessary to change the mesh connectivity. The output is a refinement `V_r`, `F_r` of the original mesh and a parametrization mesh `uv_r, FT_r`, along with target uv coordinate gradients `Du` and `Dv`. We also produce the following feature labels:
- `FE`: a $N \times 2$ matrix, where each row corresponds to an edge with the given vertex indices as endpoints. These are the target feature edges of the mesh that are aligned in the parametrization
- `ME`: like `FE`, a matrix of edges that were tagged as features in the input mesh, but are not fully aligned in the paramterization. It may be possible to improve the alignment of these edges (see below)

In [ ]:
field_params = penner.FieldParameters()
alg_params = penner.NewtonParameters()
V_r, F_r, uv_r, FT_r, FE, ME, Du, Dv = penner.parametrize_aligned(V, F, field_params, alg_params)

We provide a uv coordinate optimization that also preserves the aligned feature edges.

In order to keep the misaligned edges softly aligned, we also add a term to the optimization energy that penalizes misalignment.

In [ ]:
uv_param = penner.load_parameters("../app/symdir.json")
uv_r = penner.optimize_aligned_parameterization(V_r, F_r, uv_r, FT_r, FE, ME, Du, Dv, uv_param)

We view the optimized parametrization

In [ ]:
penner.view_seamless_parameterization(V_r, F_r, uv_r, FT_r, "feature", True)

Some edges that were identified as target feature edges may be misaligned because, in contrast with seamless constraints, many feature alignment constraints are infeasible.

We thus separate feature alignment constraints into hard constraints, which are guaranteed to be fully aligned if the method converges, and soft constraints, where we try to maximize their alignment but do not guarantee full alignment. We generally use a spanning forest of the feature graph for hard constraints, which we find to work well in most cases.

To infer seamless constraints compatible with the feature alignment constraints, we also compute a cross-field that is aligned with these feature edges, meaning that the cross field direction in faces adjacent to a feature edge is aligned with the edge direction. In order to achieve this alignment, some refinement of the input mesh is needed, e.g., to ensure no face is adjacent to multiple feature edges.

We provide a method to produce feature edges and an aligned cross field on a refined mesh:
- `V`, `F`: a possibly refined mesh
- `feature_edges`: a list of all tagged feature edges in the original mesh, represented with the endpoint vertices of the edge
- `hard_feature_edges`: a subset of the feature edges that are guaranteed to be fully aligned by our method (provided that it converges)
- `reference_field`, `theta`, `kappa`, `period_jump`: a cross field aligned to the feature edges on the refined mesh

In [ ]:
field_params = penner.FieldParameters()
field_data = penner.generate_feature_aligned_frame_field(V, F, field_params)
V, F, feature_edges, hard_feature_edges, reference_field, theta, kappa, period_jump = field_data

The following method parametrizes the mesh with the given feature and field-inferred seamless constraints. The output is a refined mesh with feature edges and a cross field on the refined surface.

Note that the set of hard feature edges generally may differ from the set of aligned feature edges in the paramterization, because the soft constraint feature edges may end up being fully aligned.

In [ ]:
alg_params = penner.NewtonParameters()
param_data = penner.generate_feature_aligned_parameterization(
  V,
  F,
  feature_edges,
  hard_feature_edges,
  reference_field,
  theta,
  kappa,
  period_jump,
  alg_params)
V_r, F_r, uv_r, FT_r, aligned_feature_edges_r, misaligned_feature_edges_r, reference_field_r, theta_r, kappa_r, period_jump_r = param_data

## Intrinsic Metric Optimization

In [ ]:
field_params = penner.FieldParameters()
reference_field, theta, kappa, period_jump = penner.generate_frame_field(V, F, field_params)
Th_hat = penner.compute_frame_field_cones(V, F, kappa, period_jump)

In [ ]:
free_cones = []
metric_params = penner.ConeMetricParameters()
cone_metric, vtx_reindex = penner.generate_cone_metric(V, F, Th_hat, free_cones, metric_params);

In [ ]:
opt_energy = penner.LogLengthEnergy(cone_metric, 2)
proj_params = penner.ProjectionParameters()
opt_params = penner.OptimizationParameters()
opt_params.num_iter = 5
cone_metric_opt = penner.optimize_metric(cone_metric, opt_energy, proj_params, opt_params);

In [ ]:
coords_opt = cone_metric_opt.get_metric_coordinates()
V_r, F_r, uv_r, FT_r = penner.parametrize_metric(V, F, cone_metric, coords_opt)

The following function can be used to view a generic parametrization.

In [ ]:
penner.view_parametrization(V_r, F_r, uv_r, FT_r, "cones", True)

In [ ]:
m = 'eight'
field_path = os.path.join(data_dir, m + '.ffield')
penner.load_frame_field(field_path)

In [ ]:
#penner.parametrize_seamless(V, F, field_params, alg_params)
penner.parametrize_seamless(V, F, field_params, alg_params)

In [ ]:
import sys, os
file_dir = '../py'
moudle_dir = os.path.join(file_dir)
sys.path.append(file_dir)
import penner
import igl
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
data_dir = os.path.join('..', '..', 'data', 'closed-Myles')
V, F = igl.read_triangle_mesh(os.path.join(data_dir, 'eight.obj'))